In [5]:
# !pip install selenium
# !pip install openpyxl
import time
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as ec
from selenium.webdriver.common.action_chains import ActionChains

In [6]:
class StocksScraper:
    def __init__(self, driver, timeout=10):
        self.driver=driver
        self.wait=WebDriverWait(self.driver,timeout=timeout)
        self.data=[]
        
    def wait_for_page_to_load(self):
            page_title=self.driver.title
            try: 
                self.wait.until(
                    lambda d: d.execute_script("return document.readyState") == "complete"
                    )
            except:
                # page_title = self.driver.title
                print(f"The page \"{page_title}\" did not get fully loaded within the given duration.\n")
            else:
                # page_title = self.driver.title
                print(f"The page is \"{page_title}\" fully loaded.\n")
                
    def access_url(self,url):
        self.driver.get(url)
        self.wait_for_page_to_load()
        
    def access_most_active_stocks(self):
        # hovering on Markets menu
        actions = ActionChains(self.driver)
        markets_menu = self.wait.until(
            ec.presence_of_element_located((By.XPATH, "/html/body/div[1]/header/div/nav/ol/li[3]/a/div"))
        )
        actions.move_to_element(markets_menu).perform()

        # hovering on stocks 
        # actions = ActionChains(driver)
        stocks_menu = self.wait.until(
            ec.presence_of_element_located((By.XPATH, "/html/body/div[1]/header/div/nav/ol/li[3]/ol/li[1]/a/span"))
        )
        actions.move_to_element(stocks_menu).perform()

        # click on trending 
        trending = self.wait.until(
            ec.element_to_be_clickable((By.XPATH, "/html/body/div[1]/header/div/nav/ol/li[3]/ol/li[1]/ol/li[4]/a/span"))
        )
        trending.click()
        time.sleep(5)
        self.wait_for_page_to_load()


        # click on most active
        most_active = self.wait.until(
            ec.element_to_be_clickable((By.XPATH, "/html[1]/body[1]/div[1]/div[4]/main[1]/section[1]/section[1]/section[1]/section[1]/section[1]/div[1]/div[1]/div[1]/a[1]"))
        )
        most_active.click()
        time.sleep(5)
        self.wait_for_page_to_load()
        
    def extract_stocks_data(self):
        while True:
            # extract data from webpage 
            self.wait.until(
                ec.presence_of_all_elements_located((By.TAG_NAME, "table"))
            )
            rows = self.driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
            time.sleep(5)
            # print(len(rows))
            
            for row in rows:
                values = row.find_elements(By.TAG_NAME, "td")
                # time.sleep(5)
                
                # print([val.text for val in values])
                
                stock = {
                    "name": values[1].text,
                    "symbol": values[0].text,
                    "price": values[3].text,
                    "change": values[4].text,
                    "volume": values[6].text,
                    "market_cap": values[8].text,
                    "pe_ratio": values[9].text
                }
                
                self.data.append(stock)

            
            # click next 
            try:
                next_button=self.wait.until(
                    ec.element_to_be_clickable((By.XPATH, "/html/body/div[1]/div[4]/main/section/section/section/section/section[1]/div/div[4]/div[3]/button[3]"))
                    )
            except: 
                print("The 'next' button is not clickable. We have navigated through all the pages.")
                break
            else:
                next_button.click() 
                time.sleep(5)
                
                
    def clean_and_save_data(self,filename="temp"):
        stocks_df=(
            pd
            .DataFrame(self.data)
            .apply(lambda col: col.str.strip() if col.dtype == "object" else col)
            .assign(
                price=lambda df: df.price.str.replace(",",""),
                change=lambda df: df.change.str.replace("+",""),
                volume= lambda df: df.volume.str.replace("M",""),
                market_cap= lambda df: df.market_cap.apply(lambda val: float(val.replace("B","")) if "B" in val else float(val.replace("T",""))*1000),
                pe_ratio= lambda df: df.pe_ratio.replace("--",np.nan).str.replace(",","")
            )
            
            # transform columns object to numeric
            .assign(
                price=lambda df: pd.to_numeric(df.price),
                change=lambda df: pd.to_numeric(df.change),
                volume=lambda df: pd.to_numeric(df.volume),
                pe_ratio=lambda df: pd.to_numeric(df.pe_ratio)
            )
            
            # rename the price column
            .rename(
                columns={
                    "price":"price_usd",
                    "volume":"volume_in_million",
                    "market_cap":"market_cap_in_billion"
                }
            )
        )

        stocks_df.to_excel(f"{filename}.xlsx", index=False)

In [7]:
if __name__=="__main__":
    driver = webdriver.Chrome()
    driver.maximize_window()
    
    url = "https://finance.yahoo.com/"
    scraper=StocksScraper(driver,10)
    
    scraper.access_url(url)
    scraper.access_most_active_stocks()
    scraper.extract_stocks_data()
    scraper.clean_and_save_data("yahoo-finance-stocks-data-1")
    
    driver.quit()

The page is "Yahoo Finance - Stock Market Live, Quotes, Business & Finance News" fully loaded.

The page is "Top Trending Stocks: US stocks with the highest interest today - Yahoo Finance" fully loaded.

The page is "Most Active Stocks: US stocks with the highest trading volume today - Yahoo Finance" fully loaded.

The 'next' button is not clickable. We have navigated through all the pages.
